# BobGuard: AI-Assisted Secure Code Analyzer

**IBM SkillsBuild Cyber Security BootCamp 2026 — Student Project**

BobGuard is a defensive educational tool that checks Python source code for selected risky coding patterns and reports findings with severity and remediation guidance. It can be developed and reviewed with IBM Bob, but it is an independent student project and does not claim an official IBM integration.

**Important:** This is a lightweight educational scanner, not a substitute for a professional security audit. Pattern-based checks can produce false positives and miss vulnerabilities.

## 1. Install dependencies
Install the packages listed in `requirements.txt`. This notebook uses Bandit, a Python security linter, alongside a few transparent educational checks.

In [ ]:
%pip install -r requirements.txt

In [ ]:
from pathlib import Path
import re
import subprocess
import sys
import json
import pandas as pd

print('BobGuard environment ready.')

## 2. Add sample code to scan
You can replace this example with a local `.py` file you own or are authorized to assess. The sample deliberately contains common insecure patterns for demonstration.

In [ ]:
sample_code = '''
import subprocess
import hashlib

password = "admin123"  # Example hard-coded secret
query = "SELECT * FROM users WHERE name = '%s'" % input("Name: ")
subprocess.call("echo " + input("Message: "), shell=True)
hash_value = hashlib.md5(b"demo").hexdigest()
'''

sample_path = Path('sample_insecure.py')
sample_path.write_text(sample_code, encoding='utf-8')
print(f'Sample created: {sample_path.resolve()}')

## 3. Run educational pattern checks
These checks are intentionally simple and should be treated as review signals, not definitive proof of a vulnerability.

In [ ]:
PATTERNS = [
    ("Hard-coded credential indicator", re.compile(r'(?i)\b(password|passwd|api_key|secret)\s*=\s*[\"\'][^\"\']+[\"\']'), "High", "Move secrets to environment variables or a secrets manager; rotate exposed credentials."),
    ("Potential SQL query string formatting", re.compile(r'(?i)(select|insert|update|delete).*(%s|\.format\(|f[\"\'])'), "Medium", "Use parameterized queries provided by your database library."),
    ("Subprocess with shell=True", re.compile(r'subprocess\.(call|run|Popen)\(.*shell\s*=\s*True'), "High", "Avoid shell=True with untrusted input; pass an argument list and validate inputs."),
    ("Weak MD5 hash use", re.compile(r'(?i)hashlib\.md5\('), "Medium", "For security-sensitive password storage, use a dedicated password hashing function such as Argon2, scrypt, or bcrypt."),
]

def scan_patterns(source_text, filename='<memory>'):
    findings = []
    for line_number, line in enumerate(source_text.splitlines(), start=1):
        for title, pattern, severity, recommendation in PATTERNS:
            if pattern.search(line):
                findings.append({
                    'file': filename,
                    'line': line_number,
                    'finding': title,
                    'severity': severity,
                    'evidence': line.strip()[:180],
                    'recommendation': recommendation
                })
    return findings

findings = scan_patterns(sample_path.read_text(encoding='utf-8'), str(sample_path))
findings_df = pd.DataFrame(findings)
findings_df

## 4. Run Bandit
Bandit performs additional static checks on Python code. The command is run only against the selected local file.

In [ ]:
def run_bandit(filepath):
    try:
        result = subprocess.run(
            [sys.executable, '-m', 'bandit', '-f', 'json', str(filepath)],
            capture_output=True, text=True, timeout=60
        )
        if result.stdout.strip():
            data = json.loads(result.stdout)
            return data.get('results', []), result.stderr.strip()
        return [], result.stderr.strip() or 'Bandit returned no JSON output.'
    except FileNotFoundError:
        return [], 'Bandit is not installed. Install requirements.txt and retry.'
    except subprocess.TimeoutExpired:
        return [], 'Bandit timed out while scanning this file.'
    except json.JSONDecodeError:
        return [], 'Could not parse Bandit output.'

bandit_results, bandit_message = run_bandit(sample_path)
print(bandit_message if bandit_message else f'Bandit findings: {len(bandit_results)}')
pd.DataFrame([{
    'file': item.get('filename'),
    'line': item.get('line_number'),
    'test_id': item.get('test_id'),
    'issue': item.get('issue_text'),
    'severity': item.get('issue_severity'),
    'confidence': item.get('issue_confidence')
} for item in bandit_results])

## 5. Scan a file you are authorized to assess
Enter a local Python file path. Do not scan systems or files without permission.

In [ ]:
target = input('Path to authorized .py file (press Enter to use sample_insecure.py): ').strip()
target_path = Path(target) if target else sample_path

if not target_path.is_file():
    print('File not found. Check the path and try again.')
elif target_path.suffix.lower() != '.py':
    print('For this prototype, choose a Python (.py) source file.')
else:
    target_source = target_path.read_text(encoding='utf-8', errors='replace')
    results = scan_patterns(target_source, str(target_path))
    report_df = pd.DataFrame(results)
    if report_df.empty:
        print('No findings from the educational pattern checks. This does not prove the code is secure.')
    else:
        display(report_df)
        output_path = Path('bobguard_report.csv')
        report_df.to_csv(output_path, index=False)
        print(f'Report saved to {output_path.resolve()}')

## 6. How IBM Bob fits into this project
Use IBM Bob as a development partner to help plan features, explain findings, write unit tests, review code for security issues, and improve documentation. Ask it to verify every generated change and explain the reasoning. This notebook itself does not call an IBM Bob API and does not automatically send source code to IBM Bob.

## 7. Limitations and future improvements
- Add unit tests and more language-aware parsing.
- Add SARIF/JSON export and a severity summary.
- Integrate approved secret-scanning and dependency-audit tools.
- Add a review workflow in which a human confirms findings before fixes are applied.
- Validate results against a documented test set and record false positives/false negatives.

**Responsible use:** Scan only code you own or have explicit permission to assess. Never put real credentials into sample files or reports.